In [13]:
!pip install sentence-transformers nlpaug numpy
!pip install torch transformers scikit-learn pandas -q

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gdown

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import nlpaug
import nlpaug.augmenter.word as naw

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import resample

# Download wordnet from nltk
import nltk
nltk.download('wordnet')


import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [15]:
data = pd.read_csv(r'/content/all_data_after_txtCorr.csv', encoding='utf-8')
data.sample(10)

,post,label
1416,"['big', 'penis', 'rigelattempts', 'n', 'n', 'l...",1
6955,"['want', 'fly', 'like', 'rocket', 'want', 'fly...",0
4433,"['like', 'somebody', 'lake', 'damn', 'skippy']",0
11553,"['haha', 'figure', 'p', 'assume', 'illegal', '...",0
7968,"['contact', 'talk', 'baby', 'bunz', 'wifey', '...",0
5234,"['doctor', 'leave', 'room', 'change', 'go', 's...",0
12442,"['one', 'specific', 'relationship', 'deal', 'b...",0
9888,"['scale', 'dirty', 'mind', 'seem', 'let', 'go'...",0
5092,"['join', 'chatroom', 'thanks']",0
299,"['overrated', 'musician', 'hmm', 'overrate', '...",1


In [16]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12834 entries, 0 to 12833
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   post    12834 non-null  object
 1   label   12834 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 200.7+ KB


In [17]:
print(f'The shape of the dataset is: {data.shape}')
print(f'The columns within the dataset are : {list(data.columns)}')
print(f'The classes within the dataset are: {list(data.label.value_counts().index)}')

The shape of the dataset is: (12834, 2)
The columns within the dataset are : ['post', 'label']
The classes within the dataset are: [0, 1]


In [18]:
# Showing the distribution of the data we have

# Prepare the data
label_counts = data['label'].value_counts().reset_index()
label_counts.columns = ['label', 'count']

# Create subplots (1 row, 2 columns)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Class Distribution (Bar Chart)", "Class Distribution (Pie Chart)"),
    specs=[[{"type": "bar"}, {"type": "pie"}]]
)

# Define colors
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# --- Bar chart ---
fig.add_trace(
    go.Bar(
        x=label_counts['label'],
        y=label_counts['count'],
        text=label_counts['count'],
        textposition='outside',
        marker_color=colors[:len(label_counts)],
        name='Class Count'
    ),
    row=1, col=1
)

# --- Pie chart ---
fig.add_trace(
    go.Pie(
        labels=label_counts['label'],
        values=label_counts['count'],
        textinfo='percent+label',
        marker=dict(colors=colors[:len(label_counts)]),
        name='Class Percentage'
    ),
    row=1, col=2
)

# --- Layout ---
fig.update_layout(
    title_text="Class Distribution Overview",
    title_x=0.5,
    showlegend=False,
    template="plotly_white",
    height=500,
    width=1000,
    title_font_size=26
)

fig.show()


In [19]:
data.drop(data[data.post == '[]'].index, axis = 0,inplace = True)

In [20]:
import pandas as pd
import random

def balance_data_by_length(data: pd.DataFrame, min_length: int = 90, seed: int = 42):

    # Convert token lists into strings for length checking
    data = data.copy()
    data['text'] = data['post'].apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))

    # Filter by minimum length
    data = data[data['text'].str.len() >= min_length]

    # Separate classes
    major_class = data[data['label'] == 0]
    minor_class = data[data['label'] == 1]

    # Get target count = minor class count
    target_count = len(minor_class)

    # Random sample from major class
    major_sample = major_class.sample(n=target_count, random_state=seed)

    # Combine and shuffle
    balanced_data = pd.concat([minor_class, major_sample]).sample(frac=1, random_state=seed).reset_index(drop=True)

    return balanced_data.drop(columns=['text'])


In [21]:
data = balance_data_by_length(data)
data

,post,label
0,"['true', 'city', 'folk', 'call', 'country', 'f...",0
1,"['answer', 'banana', 'question', 'wrong', 'ban...",1
2,"['lady', 'gaga', 'receive', 'nomination', 'org...",0
3,"['hey', 'everyone', 'follow', 'spam', 'want', ...",0
4,"['damn', 'person', 'get', 'hater', 'step', 'ne...",1
...,...,...
1505,"['like', 'pickle', 'lol', 'mean', 'dill', 'pic...",0
1506,"['home', 'alone', 'like', 'loser', 'time', 'al...",0
1507,"['hello', 'tabi', 'school', 'favorite', 'old',...",0
1508,"['size', 'matter', 'haha', 'depend', 'talk', '...",0


In [22]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.optim import AdamW

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

In [24]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import json

# ======================================================
# FOCAL LOSS
# ======================================================
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = torch.nn.functional.cross_entropy(
            logits, targets, weight=self.alpha, reduction='none'
        )
        pt = torch.exp(-ce_loss)
        loss = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


# ======================================================
# DATASET CLASS
# ======================================================
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# ======================================================
# TRAINING FUNCTION
# ======================================================
def train_imbalanced_text_classifier_full(
    data,
    text_col='post',
    label_col='label',
    model_name="distilbert-base-uncased",
    batch_size=32,
    lr=2e-5,
    epochs=5,
    max_len=128,
    patience=2,
    gamma=2.0,
    save_path="bully_model"
):

    print("🚀 Starting training...")

    # ======================================================
    # PREPARE DATA
    # ======================================================
    texts = data[text_col].apply(
        lambda x: " ".join(x) if isinstance(x, list) else str(x)
    ).tolist()

    labels = data[label_col].astype(int).tolist()

    X_train, X_val, y_train, y_val = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=42
    )

    # ======================================================
    # TOKENIZER
    # ======================================================
    tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

    train_ds = TextDataset(X_train, y_train, tokenizer, max_len)
    val_ds = TextDataset(X_val, y_val, tokenizer, max_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)

    # ======================================================
    # MODEL SETUP
    # ======================================================
    num_labels = len(np.unique(labels))

    model = DistilBertForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    class_weights = compute_class_weight(
        "balanced", classes=np.unique(labels), y=labels
    )
    alpha = torch.tensor(class_weights, dtype=torch.float).to(device)

    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn = FocalLoss(alpha=alpha, gamma=gamma)

    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None

    # ======================================================
    # TRAIN LOOP
    # ======================================================
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        # ================= VALIDATION =================
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels_batch = batch["labels"].to(device)

                outputs = model(input_ids, attention_mask=attention_mask)
                loss = loss_fn(outputs.logits, labels_batch)

                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # ================= EARLY STOPPING =================
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"⚠️ No improvement ({patience_counter}/{patience})")

            if patience_counter >= patience:
                print("🛑 Early stopping")
                break

    # Restore best model
    if best_state:
        model.load_state_dict(best_state)
        print("✅ Best model restored")

    # ======================================================
    # EVALUATION + THRESHOLD TUNING
    # ======================================================
    model.eval()
    probs, true_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            prob = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()

            probs.extend(prob)
            true_labels.extend(batch["labels"].numpy())

    probs = np.array(probs)
    true_labels = np.array(true_labels)

    best_thresh, best_f1 = 0.5, 0

    for t in np.linspace(0.1, 0.9, 17):
        preds = (probs >= t).astype(int)
        report = classification_report(true_labels, preds, output_dict=True)
        f1 = report['1']['f1-score']

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    print(f"📈 Best Threshold: {best_thresh:.2f} | F1: {best_f1:.3f}")

    final_preds = (probs >= best_thresh).astype(int)
    final_report = classification_report(true_labels, final_preds, digits=3)
    print("\n--- FINAL REPORT ---")
    print(final_report)

    # ======================================================
    # SAVE MODEL (CRITICAL FIX)
    # ======================================================
    print("💾 Saving model...")

    model.to("cpu")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    with open(f"{save_path}/threshold.json", "w") as f:
        json.dump({"threshold": float(best_thresh)}, f)

    print("✅ Model saved in Hugging Face format!")

    return model, tokenizer, best_thresh, final_report

In [25]:
model, tokenizer, best_thresh, report = train_imbalanced_text_classifier_full(data)

🚀 Starting training...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/5
Train Loss: 0.1582 | Val Loss: 0.1555
Epoch 2/5
Train Loss: 0.1293 | Val Loss: 0.1432
Epoch 3/5
Train Loss: 0.1027 | Val Loss: 0.1426
Epoch 4/5
Train Loss: 0.0847 | Val Loss: 0.1473
⚠️ No improvement (1/2)
Epoch 5/5
Train Loss: 0.0636 | Val Loss: 0.2060
⚠️ No improvement (2/2)
🛑 Early stopping
✅ Best model restored
📈 Best Threshold: 0.35 | F1: 0.731

--- FINAL REPORT ---
              precision    recall  f1-score   support

           0      0.738     0.689     0.712       151
           1      0.708     0.755     0.731       151

    accuracy                          0.722       302
   macro avg      0.723     0.722     0.722       302
weighted avg      0.723     0.722     0.722       302

💾 Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved in Hugging Face format!


In [27]:
import joblib

# Save model and tokenizer in Hugging Face format
model.save_pretrained("bully_model")
tokenizer.save_pretrained("bully_model")

# Save threshold separately as .pkl
joblib.dump(best_thresh, "bully_threshold.pkl")

print("✅ Model, tokenizer, and threshold saved successfully.")
import joblib

bundle = {
    "model_state": model.state_dict(),
    "tokenizer": tokenizer,
    "threshold": best_thresh
}
joblib.dump(bundle, "bully_model_bundle.pkl")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model, tokenizer, and threshold saved successfully.


['bully_model_bundle.pkl']

In [31]:
pip install gradio_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.2 MB/s eta 0:00:00
